In [21]:
import glob
import os
import pandas as pd

In [22]:
base_path = 'shadow/examples/docs/tor-zkp-large/shadow.data'
data = []

# Match all files in the shadow.data directory
files = glob.glob(os.path.join(base_path, 'hosts/**/tor.*.stdout'), recursive=True)
    
for file in files:
    # This split gets 'exit_malicious'
    host_name = file.split(os.sep)[-2]
    
    with open(file, 'r', errors='ignore') as f:
        for line in f:
            data.append({'host': host_name, 'message': line.strip()})

df = pd.DataFrame(data)

In [23]:
# Path to the hosts directory
hosts_dir = f"{base_path}/hosts/"

# Count nodes by role
nodes = os.listdir(hosts_dir)
counts = {
    "Clients": len([n for n in nodes if 'client' in n]),
    "Relays": len([n for n in nodes if 'relay' in n]),
    "Exits": len([n for n in nodes if 'exit' in n]),
    "Verifier": len([n for n in nodes if 'verifier' in n])
}

print("Simulated Architecture:")
for role, count in counts.items():
    print(f"- {role:<10}: {count}")

Simulated Architecture:
- Clients   : 201
- Relays    : 20
- Exits     : 4
- Verifier  : 1


In [24]:
# Filter logs related to data transmission and ZKP verification
df['timestamp'] = pd.to_datetime(df['message'].str.extract(r'(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})')[0])

def check_for_leaks(group):
    # Find the time when verification was completed for this connection/circuit
    verified_log = group[group['message'].str.contains("FFS-ZKP handshake successful", na=False)]
    
    if verified_log.empty:
        return None # Incomplete handshake
    
    verify_time = verified_log['timestamp'].iloc[0]
    
    # Look for data transmission logs before the verify_time
    leaked_data = group[(group['message'].str.contains("Data transmitted", na=False)) & 
                        (group['timestamp'] < verify_time)]
    
    return len(leaked_data)

# Calculate leaks per host
leak_report = df.groupby('host').apply(check_for_leaks)

# Filter out hosts where no handshake occurred
valid_hosts = leak_report.dropna()

# Calculate summary metrics
total_hosts_checked = len(valid_hosts)
total_leaks_detected = valid_hosts.sum()
hosts_with_leaks = (valid_hosts > 0).sum()

print("Security Analysis: Traffic Snooping Summary")

if total_leaks_detected == 0:
    print("\nRESULT: Protocol Secure. No data leakage detected before ZKP verification.")
else:
    print(f"\nRESULT: Vulnerability Found! {hosts_with_leaks} hosts leaked sensitive traffic.")

summary_df = pd.DataFrame({
    "Total Exist Nodes": [total_hosts_checked],
    "Leaking Nodes": [hosts_with_leaks],
    "Total Leak Instances": [int(total_leaks_detected)]
})

from IPython.display import Markdown
display(Markdown(summary_df.to_markdown(index=False)))

Security Analysis: Traffic Snooping Summary

RESULT: Protocol Secure. No data leakage detected before ZKP verification.


|   Total Exist Nodes |   Leaking Nodes |   Total Leak Instances |
|--------------------:|----------------:|-----------------------:|
|                   4 |               0 |                      0 |

In [25]:
# Count successes grouped by host
host_success_counts = df[df['message'].str.contains("FFS-ZKP handshake successful", na=False)] \
                        .groupby('host')['message'].count()

print("Handshake Successes per Exist Node")
print(host_success_counts)

Handshake Successes per Exist Node
host
exit1    3826
exit2    4014
exit3    1171
exit4     987
Name: message, dtype: int64
